# DubStudio AI — free browser test

Upload a short, single-speaker video, translate and dub the speech, run GPU lip-sync, and download an MP4. No installation on your Windows laptop.

**Start:** sign in to Google → Runtime → Change runtime type → select **T4 GPU** (or another available GPU) → Save. Run cells in order with their ▶ buttons. Free GPUs are not guaranteed; if none is available, try again later. Do not select a paid runtime if you want to stay free.

This runs inside the notebook, without a public web server or tunnel. The first setup downloads several GB and can take a while. Files disappear when the runtime is reset; download your result before leaving.

**Status:** notebook structure and Python syntax checked; the GPU installation and model inference have not been tested by the author. Installation can fail if Colab or upstream packages change. Model quality is not guaranteed. NLLB/MMS are noncommercial research models. Use a clearly visible, front-facing single speaker, no cuts, initially 5–10 seconds. The output uses a stock synthetic voice and replaces the original audio, including background music.


In [ ]:
# 1. Check that Google assigned a GPU
import subprocess
from pathlib import Path
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('No GPU assigned. Select Runtime > Change runtime type > GPU, then rerun.')
print(result.stdout)
ROOT = Path('/content/dubstudio-ai')
MUSE = Path('/opt/MuseTalk')


In [ ]:
# 2. Install isolated Python environments on the temporary Colab machine
import sys

def command(args, cwd=None):
    print('Running:', ' '.join(str(x) for x in args), flush=True)
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

command(['apt-get', 'update', '-qq'])
command(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'git', 'libgl1', 'libglib2.0-0'])
command([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'])
UV = [sys.executable, '-m', 'uv']
command(UV + ['python', 'install', '3.10'])
for env in ['/opt/muse', '/opt/studio']:
    if not Path(env, 'bin/python').exists():
        command(UV + ['venv', '--python', '3.10', '--seed', env])
if not ROOT.exists():
    command(['git', 'clone', 'https://github.com/seshanksenku/dubstudio-ai.git', ROOT])
# Use the reviewed first implementation, independently of later repository changes.
command(['git', 'checkout', '785134a5c9b7005f74b79d806048db3db6e780b0'], cwd=ROOT)
if not MUSE.exists():
    command(['git', 'clone', 'https://github.com/TMElyralab/MuseTalk.git', MUSE])
command(['git', 'checkout', '0a89dec45a0192b824e3cf4daf96c239440c5ed8'], cwd=MUSE)


In [ ]:
# 3. Install lip-sync and speech dependencies (large first-time download)
muse_pip = ['/opt/muse/bin/python', '-m', 'pip']
studio_pip = ['/opt/studio/bin/python', '-m', 'pip']
command(muse_pip + ['install', 'torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2', '--index-url', 'https://download.pytorch.org/whl/cu118'])
command(muse_pip + ['install', '-r', MUSE / 'requirements.txt'])
command(muse_pip + ['install', 'openmim==0.3.9'])
command(['/opt/muse/bin/mim', 'install', 'mmengine==0.10.7'])
# Prebuilt CUDA 11.8 / PyTorch 2.0 wheel avoids compiling against Colab's newer CUDA toolkit.
command(muse_pip + ['install', 'mmcv==2.0.1', '--only-binary=mmcv', '-f', 'https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html'])
command(['/opt/muse/bin/mim', 'install', 'mmdet==3.1.0'])
command(['/opt/muse/bin/mim', 'install', 'mmpose==1.1.0'])
command(studio_pip + ['install', 'torch==2.2.2', '--index-url', 'https://download.pytorch.org/whl/cpu'])
command(studio_pip + ['install', '-r', ROOT / 'requirements.txt'])
print('Dependencies installed. Continue to model download.')


In [ ]:
# 4. Download lip-sync weights and verify the GPU environment
import os
os.environ['DATA_DIR'] = '/content/dubstudio-data'
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['MUSETALK_DIR'] = str(MUSE)
os.environ['MUSETALK_PYTHON'] = '/opt/muse/bin/python'
command(['/opt/muse/bin/python', ROOT / 'scripts/download_models.py'])
command(['/opt/studio/bin/python', '-c', 'from app.pipeline import preflight; preflight(); print("GPU and lip-sync weights ready")'], cwd=ROOT)
print('Whisper, translation, and voice weights download on your first dubbing run.')


## Upload and choose your language

Upload one MP4, MOV, MKV, or WebM containing spoken audio. Start with a 5–10 second clip. This notebook limits videos to 15 seconds and 50 MB to keep the free test small. No public server is started.


In [ ]:
# 5. Upload one short video
from google.colab import files
import shutil
from uuid import uuid4
import json
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one video, then rerun this cell.')
name, content = next(iter(uploaded.items()))
suffix = Path(name).suffix.lower()
if suffix not in {'.mp4', '.mov', '.mkv', '.webm'}:
    raise ValueError('Please upload MP4, MOV, MKV, or WebM.')
if len(content) > 50 * 1024 * 1024:
    raise ValueError('For this free test, use a video under 50 MB.')
job_id = uuid4().hex
folder = Path(os.environ['DATA_DIR']) / job_id
folder.mkdir(parents=True)
input_path = folder / ('input' + suffix)
input_path.write_bytes(content)
# Remove the upload widget's extra copy if it exists.
local_upload = Path('/content') / Path(name).name
if local_upload.is_file() and local_upload != input_path:
    local_upload.unlink()
del uploaded, content
print('Video uploaded:', input_path.name)


In [ ]:
# 6. Choose languages and process (this can take several minutes)
source_language = "auto" # @param ["auto", "en", "hi", "te", "ta", "es", "fr", "de"]
target_language = "te" # @param ["en", "hi", "te", "ta", "es", "fr", "de"]
# en=English, hi=Hindi, te=Telugu, ta=Tamil, es=Spanish, fr=French, de=German
payload = {'id': job_id, 'input': input_path.name, 'source': source_language, 'target': target_language}
(folder / 'request.json').write_text(json.dumps(payload))
runner = ROOT / 'colab_run.py'
runner.write_text("""
import json
import logging
import sys
from pathlib import Path
from app import store
from app.config import DATA
from app.media import probe
from app.pipeline import process
logging.basicConfig(level=logging.INFO)
request = json.loads(Path(sys.argv[1]).read_text())
id = request.pop('id')
folder = DATA / id
duration, streams = probe(folder / request['input'])
if duration > 15:
    raise RuntimeError('Free notebook test: use a video 15 seconds or shorter.')
if not all(any(s['codec_type'] == kind for s in streams) for kind in ('video','audio')):
    raise RuntimeError('Video must contain a video track and spoken audio.')
request['duration'] = duration
store.init()
if store.get(id):
    store.remove(id)
store.add(id, request)
row = store.get(id)
original_update = store.update
def update(id, **kwargs):
    print(kwargs.get('stage', ''), str(kwargs.get('progress', 0)) + '%', flush=True)
    original_update(id, **kwargs)
store.update = update
try:
    process(row)
except Exception as exc:
    store.update(id, status='failed', stage='Processing failed', error=str(exc)[:500])
    raise
print('Completed. Run the download cell.', flush=True)
""")
command(['/opt/studio/bin/python', runner, folder / 'request.json'], cwd=ROOT)


In [ ]:
# 7. Preview and download your dubbed MP4
from IPython.display import Video, display
output = folder / 'dubbed.mp4'
if not output.is_file():
    raise RuntimeError('No completed output. Resolve the error in the previous cell first.')
display(Video(str(output), embed=True))
files.download(str(output))
transcript = folder / 'transcript.json'
if transcript.exists():
    print('Translated transcript:')
    data = json.loads(transcript.read_text())
    for segment in data['segments']:
        print(f"{segment['start']:.1f}s: {segment['translated']}")


## If a cell fails

Stop and send the error text from the failing cell. Do not repeatedly run all setup cells. A package installation failure, GPU allocation failure, or model inference error means the test is not complete.

If the GPU is unavailable, try another time. If lip-sync produces no video, check the log with the optional cell below and use a clearly visible single face. Review the entire output for translation and lip accuracy before using it.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [MuseTalk](https://github.com/TMElyralab/MuseTalk), [NLLB](https://huggingface.co/facebook/nllb-200-distilled-600M), [MMS](https://huggingface.co/facebook/mms-tts-tel).


In [ ]:
# Optional: inspect the lip-sync log after an error
log = folder / 'musetalk.log'
print(log.read_text()[-12000:] if log.exists() else 'No lip-sync log yet; the failure occurred earlier.')
